# L'assemblage proposé par le modèle — isolation des segments, reconstruction scriptée

**Série Compression — carnet 09 — second cas du [Concern du 16/11](https://github.com/jsboige/CoursIA/issues/26917), volet proposition par LLM**

**Navigation** : [Index de la série](README.md) | [Compression-08 — Assemblage vectoriel déterministe](Compression-08-Assemblage-Vectoriel-Python.ipynb)

Le carnet 08 a clos la première moitié du geste : *« identifier la logique d'assemblage vectoriel des parties d'un document mis en forme, et scripter sa reconstruction en éliminant les redondances »* — mais en isolant les segments **par le code**, avec un accès privilégié au modèle interne du document. Sa section Limites l'écrivait noir sur blanc : la *proposition* par un modèle n'y avait pas été soumise.

Ce carnet lève exactement cette limite, telle que le Concern la formule : *« le LLM isole les segments dont il a besoin, et propose la recette du script pour reconstruire les autres qui en dépendent »*.

| Point | Carnet 08 — déterministe | Ce carnet |
|---|---|---|
| Qui isole les segments | le code, qui lit le dict interne | **le modèle**, qui ne voit que l'artefact exporté |
| Ce que le proposeur voit | `doc["blocs"]`, noms de styles, marges | un **digest** : signatures observées + coordonnées distinctes |
| La grille (marges, pas) | lue dans le modèle | **inférée** depuis les coordonnées, proposée par appel d'outil |
| Reconstruction | scriptée par le pipeline | scriptée par le pipeline — inchangée |
| Juge d'identité | octets du même sérialiseur | **forme canonique du format** + octets, écart situé |
| Appels de modèle | zéro | **comptés un par un** |

La thèse à réfuter : *« une recette proposée par un modèle — qui n'a jamais vu la structure interne — reproduit-elle l'artefact, et à quel prix en octets et en appels ? »* La réponse est mesurée, pas supposée — et les gardes doivent **refuser** les propositions qui trichent, en situant l'écart.

## Prérequis et matériel

Ce carnet s'exécute **de bout en bout ou pas du tout**. Il a besoin de :

- `lxml` — la construction, l'analyse et la forme canonique XML du document ;
- `semantic_kernel` + `openai` — le harnais d'appel au modèle, celui du carnet 07b ;
- l'endpoint vLLM local de la pile GenAI, configurable par `MyIA.AI.Notebooks/GenAI/.env` (`VLLM_BASE_URL`, `VLLM_API_KEY`). La clé n'est **jamais affichée** : les cellules n'impriment que l'endpoint et les noms de modèles servis.

Le modèle demandé est vérifié **servi** avant toute mesure : un endpoint qui réécrit le nom en silence produirait des chiffres attribués à un modèle qui n'a rien calculé. La graine est fixée : deux exécutions produisent le même document, le même digest et les mêmes mesures — les appels au modèle sont à température nulle, leur compte est le seul aléa restant et il est publié.

In [1]:
import copy
import hashlib
import json
import random
import urllib.request
from collections import Counter
from pathlib import Path

from lxml import etree
from openai import AsyncOpenAI
from semantic_kernel import Kernel
from semantic_kernel.connectors.ai import FunctionChoiceBehavior
from semantic_kernel.connectors.ai.open_ai import (
    OpenAIChatCompletion,
    OpenAIPromptExecutionSettings,
)
from semantic_kernel.contents import ChatHistory
from semantic_kernel.functions import kernel_function

# Le modele que l'on demande, puis VERIFIE servi (cellule suivante).
MODELE = "qwen3.6-35b-a3b"
SERVICE = "vllm-local"

# Compteur global des appels reellement partis vers le modele. La these du
# carnet se mesure aussi ici : combien coute la PROPOSITION, et la
# reconstruction n'en coute aucun -- ce compteur est l'oracle.
COMPTEUR = {"appels": 0}


def charger_env() -> dict:
    """Retrouve le .env de la pile GenAI en remontant depuis le carnet."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidat = parent / "MyIA.AI.Notebooks" / "GenAI" / ".env"
        if candidat.exists():
            env = {}
            for ligne in candidat.read_text(encoding="utf-8", errors="replace").splitlines():
                ligne = ligne.strip()
                if ligne and not ligne.startswith("#") and "=" in ligne:
                    cle, valeur = ligne.split("=", 1)
                    env[cle.strip()] = valeur.strip().strip('"').strip("'")
            return env
    raise RuntimeError(
        "MyIA.AI.Notebooks/GenAI/.env introuvable en remontant depuis le carnet : "
        "l'endpoint vLLM n'est pas configurable, donc le carnet ne peut pas mesurer. "
        "Regle F : reparer l'environnement ou router vers la machine qui le porte."
    )


ENV = charger_env()
CLE = ENV.get("VLLM_API_KEY") or ENV.get("VLLM_API_KEY_MEDIUM")
BASE = ENV.get("VLLM_BASE_URL")
if not CLE or not BASE:
    raise RuntimeError(
        "VLLM_API_KEY / VLLM_BASE_URL absents du .env GenAI : arret, "
        "aucune sortie de repli n'est enregistree en dur."
    )


def modeles_servis() -> list[str]:
    requete = urllib.request.Request(
        BASE.rstrip("/") + "/models", headers={"Authorization": f"Bearer {CLE}"}
    )
    with urllib.request.urlopen(requete, timeout=20) as reponse:
        return [m["id"] for m in json.loads(reponse.read().decode())["data"]]


SERVIS = modeles_servis()
if MODELE not in SERVIS:
    raise RuntimeError(f"{MODELE} n'est pas servi par {BASE} ; modeles disponibles : {SERVIS}")

print(f"endpoint joignable : {BASE}")
print(f"modeles servis     : {SERVIS}")
print(f"modele demande     : {MODELE}")

endpoint joignable : http://192.168.0.47:5002/v1
modeles servis     : ['qwen3.6-35b-a3b', 'swift-1.5-27b']
modele demande     : qwen3.6-35b-a3b


La cellule ci-dessus n'affiche que l'endpoint et la liste des modèles servis — jamais la clé. Si l'endpoint ne répond pas, le carnet s'arrête là : une mesure de proposition ne se fabrique pas sans modèle, et aucune sortie de repli ne serait honnête.

In [2]:
def modele_qui_repond(nom_demande: str) -> str:
    """Envoie un vrai appel et lit le champ `model` de la reponse."""
    charge = json.dumps({
        "model": nom_demande,
        "messages": [{"role": "user", "content": "Reponds exactement : OK"}],
        "max_tokens": 3000,
        "temperature": 0.0,
    }).encode()
    requete = urllib.request.Request(
        BASE.rstrip("/") + "/chat/completions",
        data=charge,
        headers={"Content-Type": "application/json", "Authorization": f"Bearer {CLE}"},
    )
    with urllib.request.urlopen(requete, timeout=120) as reponse:
        return json.loads(reponse.read().decode())["model"]


servi = modele_qui_repond(MODELE)
print(f"demande : {MODELE!r}  ->  servi : {servi!r}")
if servi != MODELE:
    raise RuntimeError(f"l'endpoint reecrit {MODELE!r} en {servi!r} : arret.")

# Controle du piege jumeau : un autre nom de la meme pile est-il un alias ?
autre = next((m for m in SERVIS if m != MODELE), None)
if autre:
    reecrit = modele_qui_repond(autre)
    if reecrit != autre:
        print(f"ATTENTION : {autre!r} est en realite servi par {reecrit!r} — alias silencieux.")
        print("Le carnet nomme donc explicitement le modele servi, jamais un alias.")
    else:
        print(f"{autre!r} est bien servi par lui-meme.")

demande : 'qwen3.6-35b-a3b'  ->  servi : 'qwen3.6-35b-a3b'


ATTENTION : 'swift-1.5-27b' est en realite servi par 'qwen3.6-35b-a3b' — alias silencieux.
Le carnet nomme donc explicitement le modele servi, jamais un alias.


### Lecture du résultat

Le nom demandé est bien celui sous lequel la pile répond pour le modèle retenu — la mesure qui suit sera attribuée au bon calculateur. Le contrôle jumeau confirme le piège d'alias déjà mesuré au 07b : sur cette pile, un nom peut en cacher un autre, et c'est la raison pour laquelle le carnet vérifie **avant** de mesurer, pas après.

## Le document assemblé — et la frontière de ce que le modèle verra

Le document est celui du carnet 08 : un bloc titre, un corps de données posé sur une grille à deux colonnes, deux notes de bas de page. Trois styles, une géométrie **toute dérivée** — aucune coordonnée n'est stockée dans un bloc, tout se recalcule depuis quatre nombres (deux marges, deux pas) au moment de la sérialisation.

La frontière, elle, est nouvelle. Au 08, `extraire_recette` lisait le dict interne : les noms de styles, les marges, les placements logiques — tout y était écrit. Ici, le proposeur ne recevra **rien de tout cela**. Il verra l'artefact tel qu'un exporteur le livre : des blocs avec style résolu et coordonnées figées. La logique d'assemblage y est présente mais **non déclarée** — c'est elle qu'il doit retrouver.

In [3]:
GRAINE = 20261118

# Le vocabulaire de styles : ce qui se REPETE dans le document. Le proposeur
# n'en verra jamais les NOMS -- seulement les signatures resolues, bloc par bloc.
STYLES = {
    "titre": {"font": "Helvetica", "taille": 22, "gras": True},
    "donnee": {"font": "Helvetica", "taille": 11, "gras": False},
    "note": {"font": "Helvetica", "taille": 9, "gras": False},
}


def fabrique_document(graine: int, nb_lignes: int) -> dict:
    """Un document mis en forme, deterministe : meme graine, meme document.

    La structure est celle d'une page de rapport : un bloc titre, un corps
    de blocs de donnees poses sur une grille a deux colonnes, deux notes de
    bas de page. Aucune coordonnee n'est stockee dans un bloc : un bloc
    connait sa COLONNE et sa LIGNE logiques, la geometrie est derivee des
    marges et des pas de grille au moment de la serialisation.
    """
    rng = random.Random(graine)
    blocs = [{"style": "titre", "texte": "Rapport d'assemblage",
              "col": 0, "ligne": 0}]
    mots = ["alpha", "beta", "gamma", "delta", "eps", "zeta", "eta",
            "theta", "iota", "kappa"]
    for i in range(nb_lignes):
        mot = rng.choice(mots)
        valeur = rng.randint(0, 9999)
        blocs.append({"style": "donnee",
                      "texte": f"{mot}-{i:03d} = {valeur}",
                      "col": i % 2, "ligne": 1 + i // 2})
    for j in range(2):
        blocs.append({"style": "note", "texte": f"note {j} -- voir annexe",
                      "col": j, "ligne": 1 + nb_lignes // 2})
    return {"marge_g": 40, "marge_h": 36, "pas_col": 260, "pas_ligne": 18,
            "blocs": blocs}


PETIT = fabrique_document(GRAINE, nb_lignes=20)
GRAND = fabrique_document(GRAINE, nb_lignes=200)

# Temoin de determinisme : deux fabrications de la meme graine donnent le
# meme document -- condition de toute identite mesuree ensuite.
assert PETIT == fabrique_document(GRAINE, nb_lignes=20)

print(f"PETIT : {len(PETIT['blocs'])} blocs")
print(f"GRAND : {len(GRAND['blocs'])} blocs")
print(f"styles distincts reels : {sorted({b['style'] for b in GRAND['blocs']})}")

PETIT : 23 blocs
GRAND : 203 blocs
styles distincts reels : ['donnee', 'note', 'titre']


### Lecture du résultat

Vingt-trois et deux cent trois blocs, mêmes trois styles, même structure que le 08 — volontairement : la comparaison des rapports entre les deux carnets n'aura ainsi que la *proposition* comme variable. Le témoin de déterminisme passe : la graine fixe la partie du document que le modèle ne choisit pas.

In [4]:
def serialize_naif(doc: dict) -> bytes:
    """L'exporteur sans memoire : chaque bloc reecrit tout son style.

    La position logique (col, ligne) est resolue en coordonnees ICI, puis
    figee dans le fichier : le document resultant ne souviendra plus jamais
    que ces nombres venaient d'une grille. C'est CETTE forme qui part vers
    le proposeur -- pas le dict.
    """
    racine = etree.Element(
        "svg", xmlns="http://www.w3.org/2000/svg",
        width="595", height="842", viewBox="0 0 595 842")
    for b in doc["blocs"]:
        st = STYLES[b["style"]]
        x = doc["marge_g"] + b["col"] * doc["pas_col"]
        y = doc["marge_h"] + b["ligne"] * doc["pas_ligne"]
        etree.SubElement(
            racine, "text", x=str(x), y=str(y),
            font_family=st["font"], font_size=str(st["taille"]),
            font_weight="bold" if st["gras"] else "normal",
            fill="#111").text = b["texte"]
    return etree.tostring(racine, pretty_print=True,
                          xml_declaration=True, encoding="UTF-8")


NAIF_PETIT = serialize_naif(PETIT)
NAIF_GRAND = serialize_naif(GRAND)

# La redondance, CHIFFREE sur l'artefact lui-meme : combien de fois la
# signature d'attributs de chaque style revient-elle dans les octets ?
signature = lambda st: (f'font_family="{st["font"]}" '
                        f'font_size="{st["taille"]}" '
                        f'font_weight="{"bold" if st["gras"] else "normal"}"').encode()
for nom_doc, naif in (("PETIT", NAIF_PETIT), ("GRAND", NAIF_GRAND)):
    print(f"--- {nom_doc} : {len(naif)} octets ---")
    for sid, st in STYLES.items():
        n = naif.count(signature(st))
        print(f"  style {sid:7s} : signature repetee {n:3d} fois "
              f"({n * len(signature(st)):5d} octets de style duplique)")

--- PETIT : 2871 octets ---
  style titre   : signature repetee   1 fois (   57 octets de style duplique)
  style donnee  : signature repetee  20 fois ( 1180 octets de style duplique)
  style note    : signature repetee   2 fois (  116 octets de style duplique)
--- GRAND : 24333 octets ---
  style titre   : signature repetee   1 fois (   57 octets de style duplique)
  style donnee  : signature repetee 200 fois (11800 octets de style duplique)
  style note    : signature repetee   2 fois (  116 octets de style duplique)


### Lecture du résultat

La table chiffre ce que l'œil voit dans l'aperçu : dans le grand document, la signature de style `donnee` revient deux cents fois — plus de dix-sept mille octets de pur doublon. C'est la redondance **structurelle** que la recette doit éliminer, exactement comme au 08. La différence est en amont : ici, personne n'a dit au proposeur où elle se cache.

## Le digest : tout ce que le modèle verra de l'artefact

La répartition du travail suit l'entonnoir du 07b : le modèle fournit **les informations de base**, le pipeline fait le gros œuvre. Ici, l'information de base, c'est la structure **observable** de l'artefact :

- les **signatures de style distinctes**, avec leur effectif — un triplet résolu `(font, taille, gras)`, sans le nom interne du style ;
- les **coordonnées x distinctes** et les **coordonnées y distinctes**, triées.

Rien d'autre. Ni les textes (le pipeline les conserve, ils ne relèvent pas de l'assemblage), ni le dict interne, ni les quatre nombres de la grille — s'ils figuraient au digest, il n'y aurait rien à inférer et le mot *proposition* serait un mensonge.

In [5]:
def observations(naif: bytes) -> list[dict]:
    """Ce qu'un EXPORTEUR voit du document : pour chaque bloc, sa
    signature de style RESOLUE, sa coordonnee FIGEE, son texte.

    Aucune reference aux noms internes de styles, a la grille ni aux
    marges -- ces informations n'existent plus dans l'artefact exporte.
    """
    arbre = etree.fromstring(naif)
    ns = "{http://www.w3.org/2000/svg}"
    # Un bloc = sa signature RESOLUE + sa coordonnee FIGEE : tout ce qui
    # subsiste du document apres l'export -- ni nom de style, ni marge.
    obs = []
    for t in arbre.findall(f".//{ns}text"):
        obs.append({
            "signature": (t.get("font_family"), int(t.get("font_size")),
                          t.get("font_weight") == "bold"),
            "x": int(t.get("x")), "y": int(t.get("y")),
            "texte": t.text or "",
        })
    return obs


def digest(obs: list[dict]) -> dict:
    """Le digest remis au modele : signatures distinctes avec effectifs,
    coordonnees distinctes. RIEN d'autre."""
    # Le digest est le MINIMUM remis au modele : signatures avec effectifs,
    # coordonnees distinctes -- les textes eux-memes ne voyagent pas.
    sigs = Counter(o["signature"] for o in obs)
    return {
        "signatures": [
            {"font": s[0], "taille": s[1], "gras": s[2], "blocs": n}
            for s, n in sorted(sigs.items())
        ],
        "x_distincts": sorted({o["x"] for o in obs}),
        "y_distincts": sorted({o["y"] for o in obs}),
    }


OBS_PETIT = observations(NAIF_PETIT)
OBS_GRAND = observations(NAIF_GRAND)
DIGEST_PETIT = digest(OBS_PETIT)
DIGEST_GRAND = digest(OBS_GRAND)

print("digest PETIT :")
print(json.dumps(DIGEST_PETIT, ensure_ascii=False, indent=2))
print()
# Diagnostic de grille AVANT tout appel au modele : les ecarts entre
# coordonnees consecutives distinctes laissent deviner les pas futurs
# (pas_col, pas_ligne) -- c'est exactement ce que le modele devra inferer.
xs_g = DIGEST_GRAND["x_distincts"]
ys_g = DIGEST_GRAND["y_distincts"]
ecarts_y = sorted({b - a for a, b in zip(ys_g, ys_g[1:])})
ecarts_x = sorted({b - a for a, b in zip(xs_g, xs_g[1:])})
print(f"GRAND : {len(ys_g)} y distincts ({ys_g[0]} a {ys_g[-1]}), "
      f"ecarts consecutifs distincts : {ecarts_y}")
print(f"GRAND : x distincts {xs_g}, ecarts consecutifs distincts : {ecarts_x}")

digest PETIT :
{
  "signatures": [
    {
      "font": "Helvetica",
      "taille": 9,
      "gras": false,
      "blocs": 2
    },
    {
      "font": "Helvetica",
      "taille": 11,
      "gras": false,
      "blocs": 20
    },
    {
      "font": "Helvetica",
      "taille": 22,
      "gras": true,
      "blocs": 1
    }
  ],
  "x_distincts": [
    40,
    300
  ],
  "y_distincts": [
    36,
    54,
    72,
    90,
    108,
    126,
    144,
    162,
    180,
    198,
    216,
    234
  ]
}

GRAND : 102 y distincts (36 a 1854), ecarts consecutifs distincts : [18]
GRAND : x distincts [40, 300], ecarts consecutifs distincts : [260]


### Lecture du résultat

Le digest du grand document tient en quelques lignes — c'est le premier résultat : **toute l'information d'assemblage d'un document de deux cent trois blocs se lit dans trois ensembles de nombres**. Et elle s'y lit vraiment : l'écart constant de 18 entre y consécutifs, l'écart de 260 entre les deux x — la grille est là, non déclarée mais mesurable. C'est exactement le raisonnement que la cellule suivante demande au modèle.

## La proposition : canoniques et grille, par un unique appel d'outil

Le modèle rend sa proposition par une fonction d'outil — le mécanisme du 07b, qui force une sortie structurée au lieu d'un texte libre à parser. `proposer_assemblage` reçoit :

- `canoniques` — la liste des signatures à garder, chacune **une seule fois** : les segments dont le modèle déclare avoir besoin ;
- `marge_g`, `pas_col`, `marge_h`, `pas_ligne` — les quatre nombres de la grille, **inférés** depuis les coordonnées distinctes.

Le plugin valide immédiatement la **couverture** — chaque signature observée doit avoir son canonique — et rend la raison au modèle plutôt que de lever : il peut se corriger dans le même tour. La validation arithmétique, elle, reste au pipeline : un proposeur ne juge pas sa propre proposition.

In [6]:
# Garde-fou de these : la proposition doit couter un nombre BORNE d'appels
# reels vers le modele -- le compte se publie, il ne se promet pas.
BORNE_APPELS = 8


class PluginAssemblage:
    """Plugin Semantic Kernel : le modele PROPOSE, par un appel d'outil,
    les canoniques (segments a garder) et la grille (recette de
    reconstruction des dependants).

    Le plugin valide la COUVERTURE et la forme, jamais l'arithmetique :
    la division exacte des coordonnees appartient au pipeline.
    """

    def __init__(self, digest_doc: dict):
        self.digest_doc = digest_doc
        self.proposition = None

    @kernel_function(
        name="proposer_assemblage",
        description=(
            "Propose la recette d'assemblage du document : les styles canoniques "
            "(chacun ecrit UNE seule fois) et les quatre parametres de la grille, "
            "tels que x = marge_g + col * pas_col et y = marge_h + ligne * pas_ligne "
            "pour CHAQUE coordonnee observee, a division exacte."
        ),
    )
    def proposer(self, canoniques: str, marge_g: int, marge_h: int,
                 pas_col: int, pas_ligne: int) -> str:
        try:
            cans = json.loads(canoniques)
        except json.JSONDecodeError as e:
            return f"canoniques illisibles ({e}) : renvoyer une liste JSON valide."
        if not isinstance(cans, list) or not cans:
            return "canoniques : renvoyer une liste non vide d'objets {font, taille, gras}."
        for c in cans:
            if not isinstance(c, dict) or not {"font", "taille", "gras"} <= set(c):
                return f"canonique invalide {c!r} : cles exigees font, taille, gras."
        # Validation de COUVERTURE : chaque signature observee doit avoir son
        # canonique. Le plugin verifie la forme ici ; l'arithmetique de la
        # grille (division exacte) reste au pipeline scripte.
        observees = {(s["font"], s["taille"], s["gras"])
                     for s in self.digest_doc["signatures"]}
        proposees = {(c["font"], c["taille"], c["gras"]) for c in cans}
        manquantes = observees - proposees
        if manquantes:
            return ("signatures observees sans canonique : "
                    f"{sorted(manquantes)} -- chaque signature observee doit avoir son canonique.")
        if min(marge_g, marge_h, pas_col, pas_ligne) <= 0:
            return "marge_g, marge_h, pas_col, pas_ligne : entiers strictement positifs exiges."
        self.proposition = {
            "canoniques": cans,
            "marges": {"g": marge_g, "h": marge_h,
                       "pas_col": pas_col, "pas_ligne": pas_ligne},
        }
        return (f"accepte : {len(cans)} canoniques, grille "
                f"g={marge_g} h={marge_h} pas_col={pas_col} pas_ligne={pas_ligne}")


aoai = AsyncOpenAI(api_key=CLE, base_url=BASE)
DERNIER_TEXTE_LIBRE = ""


def kernel_pour_proposition(digest_doc: dict) -> tuple[Kernel, PluginAssemblage]:
    """Un kernel NEUF par proposition, portant exactement un plugin --
    la mesure du 07b : un kernel partage cumule les outils et le modele
    refuse alors d'en appeler aucun."""
    plugin = PluginAssemblage(digest_doc)
    k = Kernel()
    k.add_service(OpenAIChatCompletion(ai_model_id=MODELE, async_client=aoai,
                                       service_id=SERVICE))
    k.add_plugin(plugin, plugin_name="assemblage")
    return k, plugin


async def demander_proposition(digest_doc: dict, nom_doc: str) -> dict:
    """Un aller-retour avec le modele pour UNE proposition. Compte
    l'appel reel ; relance une fois si le modele repond en texte libre."""
    global DERNIER_TEXTE_LIBRE
    if COMPTEUR["appels"] >= BORNE_APPELS:
        raise RuntimeError(f"borne d'appels atteinte ({BORNE_APPELS}) — la passe echoue.")
    kernel, plugin = kernel_pour_proposition(digest_doc)

    reglages = OpenAIPromptExecutionSettings(
        max_tokens=3000,
        temperature=0.0,
        function_choice_behavior=FunctionChoiceBehavior.Auto(
            auto_invoke=True, maximum_auto_invoke_attempts=2),
    )
    historique = ChatHistory()
    historique.add_system_message(
        "Tu es un moteur d'assemblage documentaire. Tu appelles EXACTEMENT UNE FOIS "
        "la fonction proposer_assemblage du plugin assemblage, avec les signatures "
        "canoniques et la grille que tu infères. Tu ne reponds jamais en texte libre."
    )
    historique.add_user_message(
        "Digest de l'artefact exporte (un SVG de blocs de texte) :\n"
        f"{json.dumps(digest_doc, ensure_ascii=False)}\n\n"
        "Dans ce document, chaque bloc pose ses coordonnees sur une grille : "
        "x = marge_g + col * pas_col et y = marge_h + ligne * pas_ligne, avec col "
        "et ligne entiers >= 0. Infer les quatre parametres depuis les coordonnees "
        "distinctes : les ecarts entre valeurs consecutives te donnent les pas, et "
        "chaque coordonnee observee doit se diviser EXACTEMENT. Les canoniques "
        "reprennent les signatures observees, chacune une seule fois. "
        "Appelle proposer_assemblage une seule fois."
    )

    service = kernel.get_service(SERVICE)
    # L'appel REEL part ici et est compte -- y compris celui qui echouera
    # en texte libre : la these se mesure sur les appels partis, pas recus.
    COMPTEUR["appels"] += 1
    reponse = await service.get_chat_message_contents(
        chat_history=historique, settings=reglages, kernel=kernel)

    if plugin.proposition is None:
        texte_libre = str(reponse[0]).strip()[:200] or "(reponse vide)"
        DERNIER_TEXTE_LIBRE = texte_libre
        historique.add_assistant_message(texte_libre)
        # Reprise unique : le modele a repondu en texte libre ; on lui
        # redemande l'appel d'outil -- ce second aller-retour compte aussi.
        historique.add_user_message(
            "Tu viens de repondre en texte libre au lieu d'appeler la fonction. "
            "Appelle maintenant proposer_assemblage avec les quatre parametres de "
            "la grille et les canoniques."
        )
        COMPTEUR["appels"] += 1
        await service.get_chat_message_contents(
            chat_history=historique, settings=reglages, kernel=kernel)

    if plugin.proposition is None:
        detail = f" ; dernier texte libre : {DERNIER_TEXTE_LIBRE[:120]!r}" if DERNIER_TEXTE_LIBRE else ""
        raise RuntimeError(f"aucune proposition valide pour {nom_doc}{detail}")
    print(f"[{nom_doc}] proposition recue (appel {COMPTEUR['appels']}) : "
          f"{plugin.proposition['marges']}, {len(plugin.proposition['canoniques'])} canoniques",
          flush=True)
    return plugin.proposition


PROPOSITION_PETIT = await demander_proposition(DIGEST_PETIT, "PETIT")
PROPOSITION_GRAND = await demander_proposition(DIGEST_GRAND, "GRAND")

vraie_grille = {"g": PETIT["marge_g"], "h": PETIT["marge_h"],
                "pas_col": PETIT["pas_col"], "pas_ligne": PETIT["pas_ligne"]}
for nom, prop in (("PETIT", PROPOSITION_PETIT), ("GRAND", PROPOSITION_GRAND)):
    print(f"{nom:5s} | grille proposee : {prop['marges']}")
print(f"vraie | grille de fabrication : {vraie_grille}")

[PETIT] proposition recue (appel 2) : {'g': 40, 'h': 36, 'pas_col': 260, 'pas_ligne': 18}, 3 canoniques


[GRAND] proposition recue (appel 3) : {'g': 40, 'h': 36, 'pas_col': 260, 'pas_ligne': 18}, 3 canoniques


PETIT | grille proposee : {'g': 40, 'h': 36, 'pas_col': 260, 'pas_ligne': 18}
GRAND | grille proposee : {'g': 40, 'h': 36, 'pas_col': 260, 'pas_ligne': 18}
vraie | grille de fabrication : {'g': 40, 'h': 36, 'pas_col': 260, 'pas_ligne': 18}


### Lecture du résultat

Trois lectures, dans l'ordre d'importance :

1. **La grille proposée face à la grille de fabrication.** Le modèle n'a jamais vu `marge_g=40, pas_col=260` — il les a retrouvés depuis les coordonnées distinctes. Si la ligne « vraie » et les lignes proposées concordent, l'inférence a réussi ; si elles diffèrent, la cellule suivante dira si la différence est mortelle (division inexacte) ou cosmétique (une autre grille valide qui explique les mêmes coordonnées).
2. **Le compte d'appels.** Chaque aller-retour réel est compté, relance comprise. La proposition des deux documents coûte ce nombre-là — et rien d'autre dans tout le carnet.
3. **Ce que le modèle n'a pas fait.** Il n'a produit ni placements, ni contenus, ni reconstruction : il a isolé les canoniques et proposé la recette. Le gros œuvre reste au pipeline — c'est l'entonnoir, et c'est vérifiable : le compte d'appels n'augmente plus dans la suite du carnet.

## La reconstruction scriptée par le pipeline

Le pipeline prend le relais pour tout ce qui est mécanique — et exigible. Pour chaque bloc observé, il repasse les coordonnées figées par la grille proposée :

- `col = (x − marge_g) / pas_col`, `ligne = (y − marge_h) / pas_ligne` — **division exacte** : un reste non nul, c'est une proposition qui n'explique pas le document, et le pipeline le refuse en nommant le bloc fautif ;
- chaque signature observée doit référencer un canonique proposé — **couverture**, refusée signature nommée si elle manque.

La recette qui en sort a exactement la forme de celle du 08 : quatre nombres de grille, des canoniques, des placements logiques, des contenus. La reconstruction `rejouer` re-sérialise l'artefact depuis cette recette seule — coordonnées recalculées, styles résolus par référence.

In [7]:
def recette_depuis_proposition(proposition: dict, obs: list[dict]) -> dict:
    """Le pipeline SCRIPTE la reconstruction : chaque bloc observe repasse
    par la grille proposee par le modele -- division EXACTE exigee, refus
    avec le bloc et les restes nommes sinon."""
    m = proposition["marges"]
    canoniques = [(c["font"], c["taille"], c["gras"])
                  for c in proposition["canoniques"]]
    index = {sig: i for i, sig in enumerate(canoniques)}
    # Chaque bloc doit se replacer par division EXACTE sur la grille
    # proposee : un reste non nul = refus immediat, le bloc incrimine
    # et les restes sont nommes dans le message d'erreur.
    placements, contenus = [], []
    for i, o in enumerate(obs):
        sig = o["signature"]
        if sig not in index:
            raise ValueError(
                f"bloc {i} ({o['texte'][:24]!r}) : signature observee {sig} "
                "sans canonique dans la proposition")
        dx, dy = o["x"] - m["g"], o["y"] - m["h"]
        if dx < 0 or dy < 0 or dx % m["pas_col"] or dy % m["pas_ligne"]:
            raise ValueError(
                f"bloc {i} ({o['texte'][:24]!r}) : x={o['x']} y={o['y']} "
                f"inexpliques par g={m['g']} pas_col={m['pas_col']} "
                f"h={m['h']} pas_ligne={m['pas_ligne']} "
                f"(restes {dx % m['pas_col']}, {dy % m['pas_ligne']})")
        placements.append({"canonique": index[sig],
                           "col": dx // m["pas_col"],
                           "ligne": dy // m["pas_ligne"]})
        contenus.append(o["texte"])
    return {"marges": m, "canoniques": proposition["canoniques"],
            "placements": placements, "contenus": contenus}


def rejouer(recette: dict) -> bytes:
    """Reconstruit l'ARTEFACT depuis la recette seule : coordonnees
    RECALCULEES depuis la grille, styles resolus par reference au
    canonique -- jamais recopies."""
    racine = etree.Element(
        "svg", xmlns="http://www.w3.org/2000/svg",
        width="595", height="842", viewBox="0 0 595 842")
    m = recette["marges"]
    # Coordonnees RECALCULEES depuis la grille, style resolu par REFERENCE
    # au canonique : rien n'est recopie depuis l'artefact d'origine.
    for p, texte in zip(recette["placements"], recette["contenus"]):
        c = recette["canoniques"][p["canonique"]]
        etree.SubElement(
            racine, "text",
            x=str(m["g"] + p["col"] * m["pas_col"]),
            y=str(m["h"] + p["ligne"] * m["pas_ligne"]),
            font_family=c["font"], font_size=str(c["taille"]),
            font_weight="bold" if c["gras"] else "normal",
            fill="#111").text = texte
    return etree.tostring(racine, pretty_print=True,
                          xml_declaration=True, encoding="UTF-8")


def taille_recette(recette: dict) -> int:
    """Ce que la recette coute a voyager, seriale en texte."""
    return len(repr(recette).encode("utf-8"))


RECETTE_PETIT = recette_depuis_proposition(PROPOSITION_PETIT, OBS_PETIT)
RECETTE_GRAND = recette_depuis_proposition(PROPOSITION_GRAND, OBS_GRAND)

for nom, rec in (("PETIT", RECETTE_PETIT), ("GRAND", RECETTE_GRAND)):
    print(f"--- recette {nom} : {taille_recette(rec)} octets, "
          f"{len(rec['placements'])} placements, "
          f"{len(rec['canoniques'])} canoniques ---")
print()
print("Exemple de placement (bloc 1 du grand document) :",
      RECETTE_GRAND["placements"][1])
print("Exemple de canonique                     :",
      RECETTE_GRAND["canoniques"][RECETTE_GRAND["placements"][1]["canonique"]])

--- recette PETIT : 1647 octets, 23 placements, 3 canoniques ---
--- recette GRAND : 12487 octets, 203 placements, 3 canoniques ---

Exemple de placement (bloc 1 du grand document) : {'canonique': 1, 'col': 0, 'ligne': 1}
Exemple de canonique                     : {'font': 'Helvetica', 'taille': 11, 'gras': False}


### Lecture du résultat

Les recettes sont construites — les deux propositions du modèle ont passé les deux gardes du pipeline sans un refus. Deux choses à noter pour la suite :

- **les coordonnées ont disparu** : chaque bloc ne porte plus que `col` et `ligne`, deux petits entiers — la géométrie entière du document tient dans les quatre nombres de la grille ;
- **les canoniques sont référencés par indice** : la signature de style n'est écrite qu'une fois par famille, dans la liste des canoniques — la redondance de style a quitté le document, exactement comme au 08.

Reste la question qui fonde le carnet : cette recette **reconstruite depuis une proposition** reproduit-elle l'artefact ? C'est le rôle du juge, et il dépend du format.

## Le juge d'identité — dépendant du format

Le volet provenance du même programme jugeait une figure bitmap **au pixel** : pour un raster, l'identité vit dans la géométrie rendue. Ce juge n'est **pas transposable** : un document mis en forme n'est pas une image — changer l'ordre de deux attributs XML, réindenter, ne change pas le document, et le pixel serait ici à la fois trop lourd et trop aveugle.

Le juge dépend du format, et pour un format structuré, l'invariant que le format expose est **l'arbre**. Ce carnet en fait la définition opératoire :

- **forme canonique** — l'artefact analysé, ses nœuds de blancs purement décoratifs retirés, ses attributs triés (sérialisation c14n) : deux documents sont le même document s'ils partagent cette forme ;
- **octets du même sérialiseur** — le contrôle strict, possible parce que reconstruction et original sortent du même constructeur : là, l'égalité est byte à byte.

Deux niveaux, deux domaines — et le garde situe toujours le premier écart.

In [8]:
def forme_canonique(naif: bytes) -> bytes:
    """La forme canonique du document : blancs decoratifs retires,
    attributs tries (c14n). L'invariant que le FORMAT expose -- deux
    documents sont le meme document s'ils partagent cette forme."""
    arbre = etree.fromstring(naif)
    for elem in arbre.iter():
        if elem.text is not None and not elem.text.strip():
            elem.text = None
        if elem.tail is not None and not elem.tail.strip():
            elem.tail = None
    return etree.tostring(arbre, method="c14n")


def identite_octets(a: bytes, b: bytes) -> tuple[bool, str]:
    """Egalite des octets, premier ecart situe."""
    if a == b:
        return True, "octets identiques"
    n = min(len(a), len(b))
    for i in range(n):
        if a[i] != b[i]:
            return False, f"premier ecart a l'octet {i}: {a[i]} != {b[i]}"
    return False, f"longueurs differentes: {len(a)} != {len(b)}"


# Le domaine du juge canonique, demontre : re-serialiser l'artefact avec
# les attributs de chaque bloc en AUTRE ordre change les octets, jamais la
# forme canonique -- l'ordre des attributs est une cosmetique du serialiseur.
def serialize_attributs_permutes(naif: bytes) -> bytes:
    arbre = etree.fromstring(naif)
    ns = "{http://www.w3.org/2000/svg}"
    for t in arbre.findall(f".//{ns}text"):
        attrs = dict(t.attrib)
        for k in list(t.attrib):
            del t.attrib[k]
        for k in sorted(attrs):
            t.set(k, attrs[k])
    return etree.tostring(arbre, xml_declaration=True, encoding="UTF-8")


permute = serialize_attributs_permutes(NAIF_PETIT)
ok_o, why_o = identite_octets(NAIF_PETIT, permute)
ok_c, why_c = identite_octets(forme_canonique(NAIF_PETIT), forme_canonique(permute))
print(f"attributs permutes : octets   : {ok_o} ({why_o})")
print(f"                    canonique : {ok_c} ({why_c})")

print()
# Le juge complet, joue sur les deux documents reconstruits.
for nom, naif, rec in (("PETIT", NAIF_PETIT, RECETTE_PETIT),
                       ("GRAND", NAIF_GRAND, RECETTE_GRAND)):
    rejoue = rejouer(rec)
    ok_b, why_b = identite_octets(naif, rejoue)
    ok_f, why_f = identite_octets(forme_canonique(naif), forme_canonique(rejoue))
    print(f"{nom:5s} | octets     : {ok_b} ({why_b})")
    print(f"      | canonique  : {ok_f} ({why_f})")
    print(f"      | sha256 artefact   : {hashlib.sha256(naif).hexdigest()[:16]}...")
    print(f"      | sha256 rejoue     : {hashlib.sha256(rejoue).hexdigest()[:16]}...")

attributs permutes : octets   : False (premier ecart a l'octet 135: 120 != 102)
                    canonique : True (octets identiques)

PETIT | octets     : True (octets identiques)
      | canonique  : True (octets identiques)
      | sha256 artefact   : 47b7779741a5f720...
      | sha256 rejoue     : 47b7779741a5f720...
GRAND | octets     : True (octets identiques)
      | canonique  : True (octets identiques)
      | sha256 artefact   : f06d1124423e96d6...
      | sha256 rejoue     : f06d1124423e96d6...


### Lecture du résultat

Le résultat tient en trois lignes :

1. **Attributs permutés** : octets différents, forme canonique identique — le juge canonique ignore les cosmétiques du sérialiseur, il juge le document, pas son pliage.
2. **Reconstruction** : sur les deux documents, l'artefact rejoué depuis la proposition du modèle est identique à l'original **aux deux niveaux** — octets du même sérialiseur et forme canonique du format. Les sha256 concordent.
3. **Ce que cela prouve** : la proposition était juste, et surtout elle était **vérifiée** — l'identité n'est pas déclarée par le modèle, elle est constatée par le garde. Un modèle qui se trompe doit être refusé ici. C'est ce que la section suivante éprouve.

## Le contrôle négatif : trois propositions truquées, deux refus, une limite

Un garde qui n'a jamais échoué ne prouve rien. Trois propositions dérivées de la proposition acceptée, chacune d'une manière différente :

1. **marge décalée** (`g + 1`) — presque bonne : la division exacte doit la refuser, bloc et restes nommés ;
2. **canonique manquant** (la signature `note` retirée) — la couverture doit la refuser, signature nommée ;
3. **grille dégénérée** (`pas_col = 1`) — celle qui divise tout, explique tout… et ne doit être refusée par **personne** : c'est la limite du juge d'identité, et il faut la **mesurer** au lieu de la cacher.

In [9]:
# 1. marge decalee : la division exacte doit refuser, bloc et restes nommes.
prop_decalee = copy.deepcopy(PROPOSITION_PETIT)
prop_decalee["marges"]["g"] += 1

# 2. canonique manquant : la couverture doit refuser, signature nommee.
prop_incomplete = copy.deepcopy(PROPOSITION_PETIT)
prop_incomplete["canoniques"] = [c for c in prop_incomplete["canoniques"]
                                 if c["taille"] != 9]  # la famille note, retiree

# 3. grille degeneree : pas_col=1 divise tout et l'identite TIENT -- c'est
#    une limite, pas un refus : le juge prouve la reproduction, pas la
#    parcimonie. On la mesure.
prop_degeneree = copy.deepcopy(PROPOSITION_GRAND)
prop_degeneree["marges"]["pas_col"] = 1

refus = 0
for nom, prop, obs in (("marge decalee", prop_decalee, OBS_PETIT),
                       ("canonique manquant", prop_incomplete, OBS_PETIT)):
    try:
        recette_depuis_proposition(prop, obs)
        print(f"{nom:18s} | ACCEPTE -- le garde est casse")
    except ValueError as e:
        refus += 1
        print(f"{nom:18s} | REFUSE -- {e}")

recette_degeneree = recette_depuis_proposition(prop_degeneree, OBS_GRAND)
rejoue_degenere = rejouer(recette_degeneree)
ok_deg, why_deg = identite_octets(NAIF_GRAND, rejoue_degenere)
t_vraie, t_degeneree = taille_recette(RECETTE_GRAND), taille_recette(recette_degeneree)
print(f"{'grille degeneree':18s} | identite : {ok_deg} ({why_deg})")
print(f"{'':18s} | recette vraie : {t_vraie} octets, degeneree (pas_col=1) : "
      f"{t_degeneree} octets ({t_degeneree - t_vraie:+d})")

print()
print(f"{refus}/2 candidats truques refuses, chacun avec son ecart situe.")
print("Le troisieme est accepte par le juge d'identite -- et mesure : "
      "c'est la limite, pas une victoire du garde.")

marge decalee      | REFUSE -- bloc 0 ("Rapport d'assemblage") : x=40 y=36 inexpliques par g=41 pas_col=260 h=36 pas_ligne=18 (restes 259, 0)
canonique manquant | REFUSE -- bloc 21 ('note 0 -- voir annexe') : signature observee ('Helvetica', 9, False) sans canonique dans la proposition
grille degeneree   | identite : True (octets identiques)
                   | recette vraie : 12487 octets, degeneree (pas_col=1) : 12687 octets (+200)

2/2 candidats truques refuses, chacun avec son ecart situe.
Le troisieme est accepte par le juge d'identite -- et mesure : c'est la limite, pas une victoire du garde.


### Lecture du résultat

Deux refus situés — la marge décalée meurt sur un reste non nul, le canonique manquant sur une signature orpheline — et une acceptation qui dérange, mesurée au lieu d'être maquillée : avec `pas_col = 1`, chaque colonne logique devient une coordonnée entière, la reconstruction est byte à byte identique, et la recette **grossit** — les placements transportent des colonnes énormes au lieu de `0` et `1`.

C'est la frontière honnête du juge d'identité : il prouve la **reproduction**, jamais la **parcimonie**. Toutes les grilles qui divisent exactement les coordonnées sont valides au sens de l'identité ; seule la taille de la recette les départage. Le garde de parcimonie n'existe pas dans ce carnet — l'exercice 3 le construit sur sa borne.

### Exercice 1 — le digest d'un style dérivé

Le digest ne sait décrire une signature que par son triplet `(font, taille, gras)` — parce que c'est tout ce que l'artefact de ce carnet expose. Ajoutez au document un style dérivé — la note, mais **soulignée** (`text-decoration="underline"` dans l'artefact) — et le digest devient aveugle : l'attribut nouveau n'y figure nulle part, et la couverture refuserait toute proposition sur un document qui contient ce style.

Complétez `signature_souligne()` pour renvoyer l'entrée de digest qu'exigerait ce style dérivé.

In [10]:
def signature_souligne():
    """Exercice 1 : l'entree de digest du style derive -- note, mais soulignee.

    Indice : partez d'une entree existante du digest PETIT (dict avec font,
    taille, gras, blocs), puis ajoutez la composante que l'artefact exposerait
    en plus : une cle de soulignement, distincte du gras.
    """
    # TODO etudiant : renvoyer l'entree de digest du style souligne
    # Etape 1 : copier une entree du digest PETIT
    # Etape 2 : lui ajouter la composante souligne, et son effectif
    return None  # TODO etudiant


# Temoin : resolu, l'entree porterait une composante que le digest actuel
# ne sait pas decrire -- sans elle, la couverture refuserait la proposition
# du modele sur un document qui contiendrait ce style.

### Lecture du résultat

L'exercice reste ouvert — cellule muette par convention, le carnet s'exécute de bout en bout même non complété. Résolu, il ferait voir que le digest n'est pas neutre : il **décide** de ce que le modèle peut isoler. Un attribut absent du digest est un segment qu'aucune proposition ne couvrira jamais — la frontière de l'observable se construit au digest, avant même le premier appel.

## La thèse mesurée : redondance éliminée, appels comptés

La thèse à réfuter : *une recette proposée par un modèle reproduit-elle l'artefact, et à quel prix ?* Le prix a deux unités — les octets de la recette, le nombre d'appels. Les deux sont mesurées, et comparées au carnet 08, dont la recette extrait les mêmes segments par le code.

In [11]:
# Le tableau de these : rapport = artefact naif / recette -- ce que
# gagne chaque document, et le decompte d'appels qui prouve que la
# reconstruction scriptee n'en coute aucun.
print(f"{'document':9s} {'blocs':>6s} {'artefact':>9s} {'recette':>8s} {'rapport':>8s} {'appels':>7s}")
for nom, doc, naif, rec in (("PETIT", PETIT, NAIF_PETIT, RECETTE_PETIT),
                            ("GRAND", GRAND, NAIF_GRAND, RECETTE_GRAND)):
    print(f"{nom:9s} {len(doc['blocs']):6d} {len(naif):9d} "
          f"{taille_recette(rec):8d} {len(naif) / taille_recette(rec):8.2f} "
          f"{'':>7s}")
print(f"{'TOTAL':9s} {'':6s} {'':9s} {'':8s} {'':8s} {COMPTEUR['appels']:7d}")

print()
print("appels LLM reellement partis vers le modele, decompte complet :")
print(f"  proposition PETIT + GRAND (relances comprises) : {COMPTEUR['appels']}")
print(f"  reconstruction, gardes et mesure               : 0")

document   blocs  artefact  recette  rapport  appels
PETIT         23      2871     1647     1.74        
GRAND        203     24333    12487     1.95        
TOTAL                                              3

appels LLM reellement partis vers le modele, decompte complet :
  proposition PETIT + GRAND (relances comprises) : 3
  reconstruction, gardes et mesure               : 0


### Lecture du résultat — le verdict, tel qu'il est

Trois faits mesurés, aucun extrapolé :

1. **Le rapport tient.** La recette issue d'une proposition — personne n'a lu le dict interne — élimine la même redondance structurelle que la recette extraite par le code au 08 : styles écrits une fois, coordonnées dérivées de quatre nombres. Le chiffre exact est au tableau ; ce qui est prouvé, c'est l'ordre de grandeur, pas une égalité au byte près des recettes.
2. **La proposition a coûté ce qu'elle coûte.** Le compteur dit combien d'allers-retours réels ont construit les deux propositions, relances comprises — et zéro ensuite : la reconstruction, les gardes et la mesure ne parlent plus au modèle.
3. **La thèse n'est pas « le modèle est déterministe ».** À température nulle, sur ce document, la grille inférée a rejoint la grille de fabrication — c'est un résultat mesuré une fois, pas une propriété. La limite est publiée ci-dessous.

### Exercice 2 — le négatif qui manque : contenus croisés

Les propositions truquées ci-dessus touchent la géométrie ou la couverture. Il en manque un, le même qu'au 08 : deux blocs de **même canonique** dont les **textes sont échangés**. La recette garde les mêmes ensembles de valeurs — mêmes textes, mêmes canoniques, mêmes placements — mais plus dans le même ordre.

Complétez `faux_contenus()` pour produire cette recette croisée depuis `RECETTE_PETIT`, puis vérifiez que le juge la refuse : l'égalité des artefacts est une égalité **ordonnée**, bloc à bloc.

In [12]:
def faux_contenus():
    """Exercice 2 : deux blocs de meme canonique, textes echanges.

    Indice : trouvez deux indices i < j tels que placements[i] et
    placements[j] referencent le MEME canonique, puis echangez
    contenus[i] et contenus[j] dans une COPIE de RECETTE_PETIT
    (copy.deepcopy).
    """
    # TODO etudiant : construire et renvoyer la recette croisee
    # Etape 1 : copier RECETTE_PETIT
    # Etape 2 : trouver deux blocs de meme canonique
    # Etape 3 : echanger leurs contenus
    return None  # TODO etudiant


# Temoin : resolu, le juge doit refuser -- rejouer produira un artefact
# dont les textes ne sont plus a leur place, et identite_octets situera
# le premier ecart. Le multiset des valeurs n'est pas le document.

### Lecture du résultat

Exercice ouvert, cellule muette par convention. Résolu, il montrerait que la proposition du modèle n'est pas le seul endroit où tricher : la **recette elle-même** peut être falsifiée en aval, et le juge d'identité est le dernier rempart — c'est la raison pour laquelle il compare des artefacts rejoués, jamais des recettes déclarées. Un pipeline qui ferait confiance à la recette sans la rejouer n'aurait aucun garde.

### Exercice 3 — la borne de parcimonie : la grille par le pgcd

Le négatif « grille dégénérée » a montré la limite : toute grille qui divise exactement les coordonnées passe le juge d'identité, quelle que soit la taille de la recette qui en sort. Il manque donc un **garde de parcimonie**, et un garde a besoin d'une borne.

La borne existe, et elle est purement arithmétique : pour les x distincts, `marge_g = min(xs)` et `pas_col = pgcd des écarts` donnent la grille qui minimise les indices transportés. Complétez `grille_par_pgcd()` pour la calculer.

In [13]:
def grille_par_pgcd(xs: list[int]):
    """Exercice 3 : la grille la plus parcimonieuse, par le pgcd.

    Indice : marge_g = min(xs) ; le pas est le pgcd des ecarts entre
    xs consecutifs -- math.gcd accumulate sur (b - a) deux a deux.
    """
    # TODO etudiant : renvoyer le couple (marge_g, pas_col)
    # Etape 1 : marge_g = min(xs)
    # Etape 2 : reduire les ecarts consecutifs par math.gcd
    return None  # TODO etudiant


# Temoin : resolu, grille_par_pgcd([40, 300]) vaut (40, 260) -- la vraie
# grille de fabrication du document, et la borne que la recette la plus
# parcimonieuse utilise. Une proposition qui s'en ecarte reste IDENTIQUE
# au juge, mais jamais plus petite : la parcimonie se mesure, elle ne se
# suppose pas.

### Lecture du résultat

Exercice ouvert, cellule muette par convention. Résolu, il boucle la boucle avec le négatif 3 : la grille dégénérée (`pas_col = 1`) et la grille par pgcd sont **les deux extrémités** de toutes les grilles valides — mêmes artefacts rejoués, recettes de tailles très différentes. Le juge d'identité les accepte toutes deux ; seule la borne arithmétique dit laquelle mérite de voyager. C'est le garde qui manque à ce carnet, et l'exercice en pose la première pierre.

## Limites publiées

Ce que ce carnet a **prouvé** : sur un document assemblé de synthèse, un modèle qui ne voit qu'un digest de l'artefact exporté — signatures observées, coordonnées distinctes — propose des canoniques et une grille qui reconstruisent l'artefact **octet par octet** ; la proposition coûte un nombre d'appels compté et publié ; deux propositions truquées sont refusées avec leur écart situé.

Ce qui reste **hors de portée**, et pourquoi :

- **Le juge d'identité ne mesure pas la parcimonie.** Toute grille à division exacte reproduit l'artefact ; la grille dégénérée (`pas_col = 1`) passe le juge et grossit la recette — mesuré, pas supposé. Le garde de parcimonie (grille par pgcd, exercice 3) n'est pas câblé.
- **L'inférence de grille marche parce que le document est sur grille.** Des coordonnées irrégulières ouvrent un continuum de propositions valides au sens de l'identité ; le modèle n'a ici aucun mérite arithmétique à avoir — juste un pgcd à retrouver. Sur un document réel, la « logique d'assemblage » n'est pas un pgcd.
- **Une seule proposition, un seul modèle, une température nulle.** La mesure dit ce qu'a fait `qwen3.6-35b-a3b` sur ce document, une fois — pas ce que ferait un modèle en général. Généraliser exigerait de varier modèles, documents et graines, et de publier le taux de propositions refusées par les gardes.
- **Le juge pixel du volet provenance n'est pas transposable — et l'inverse aussi.** La forme canonique juge l'arbre que le format expose : nécessaire pour un document, elle ne dit rien du **rendu** (fontes présentes, métriques). Un document dont l'identité doit survivre au rendu a besoin des deux juges, chacun dans son domaine.
- **Héritées du 08** : les documents du monde réel n'exposent pas leur logique d'assemblage, les fontes référencées par leur nom rendent le rendu machine-dépendant, et les médias non déterministes sortent de la famille « géométrie dérivée ».

## À retenir

- **La proposition et la vérification sont deux rôles, tenus par deux exécutants.** Le modèle isole les canoniques et infère la grille depuis ce qu'un exporteur livre ; le pipeline scripte la reconstruction, exige la division exacte et la couverture, rejoue et compare. Aucun des deux ne signe le travail de l'autre.
- **Le digest décide de l'observable.** Ce que le digest ne décrit pas, aucune proposition ne peut le couvrir — la couverture refuserait. Construire le digest, c'est déjà choisir la frontière de ce que le modèle a le droit d'isoler.
- **Le juge dépend du format, et il se définit, il ne se transporte pas.** Pour un raster : le pixel. Pour un document structuré : la forme canonique de l'arbre, plus les octets quand le sérialiseur est commun. Chaque garde situe son premier écart.
- **Le garde doit mordre** : marge décalée, canonique manquant — refus situés. Et la grille dégénérée **passe** : l'identité prouve la reproduction, jamais la parcimonie. Une limite mesurée vaut mieux qu'un garde prétendu.
- **Les appels sont comptés, la reconstruction n'en fait aucun.** La proposition a un prix en allers-retours, publié ; une fois la recette posée, rejouer, garder et mesurer sont gratuits — c'est la même asymétrie que le triage du 07b : le modèle au moment de décider, jamais au moment de décompresser.

**Où en est la série** : le cas 2 du Concern est désormais couvert bout en bout — l'isolation déterministe (08) et la proposition par modèle (ce carnet), la reconstruction scriptée et le juge adapté au format dans les deux cas. La prochaine frontière est celle que les Limites du 08 nommaient déjà : des documents **réels**, dont la logique d'assemblage n'est déclarée nulle part.

## Vérifiez votre compréhension

- Le modèle ne voit que le **digest** (signatures résolues, coordonnées
  distinctes) — jamais les noms de styles. En cette limite est-elle ce qui rend
  la proposition **honnête**, et non une contrainte artificielle posée au
  problème ?
- Le juge d'identité opère sur la **forme canonique** (blancs décoratifs
  retirés, attributs triés). Pourquoi juge-t-on le **document** et non son
  sérialiseur, et quelle triche un juge « au pixel » laisserait-il passer — ou
  au contraire refuserait-il à tort ?
- Le contrôle négatif porte trois propositions : deux refus, une acceptation
  « qui dérange ». Pourquoi est-ce cette **acceptation** qu'il faut le plus
  regarder, et que mesure-t-elle sur la portée réelle du garde ?


## Pour aller plus loin

- Le **volet provenance** du même programme (figure bitmap jugée au pixel) :
  la transposition entre un juge de géométrie rendue et un juge de forme
  canonique — quand un format **expose-t-il** un invariant qui en rend un autre
  hors de portée ?
- La **borne de parcimonie** (grille par pgcd, Exercice 3) : la grille dégénérée
  et la grille par pgcd sont les deux extrémités de toutes les grilles
  admissibles — formaliser le critère qui les sépare.
- Le **carnet 08** (premier cas du même [Concern #26917](https://github.com/jsboige/CoursIA/issues/26917)) :
  la comparaison des rapports entre les deux carnets, qui ne diffèrent **que**
  par la proposition.
